# Setup

In [2]:
# setup
import os
import io
import base64
import sys
import glob
import errno
from collections import defaultdict
import numpy as np
import h5py
import scipy as sp
import itertools
import multiprocessing as mproc
import pandas as pd
import tensorflow as tf
import numba as nb

# os.environ['NUMBAPRO_NVVM'] = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.8\nvvm\bin\nvvm64_40_0.dll'
# os.environ['NUMBAPRO_LIBDEVICE'] = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.8\nvvm\libdevice'

%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML, Math, Latex
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

import matplotlib.pyplot as plt
# from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import matplotlib as mpl
# from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
#%matplotlib notebook
    
# mpl.rcParams['text.usetex'] = 'True'
mpl.rcParams['axes.grid'] = False
mpl.rcParams['xtick.top'] = True
mpl.rcParams['xtick.bottom'] = True
mpl.rcParams['ytick.left'] = True
mpl.rcParams['ytick.right'] = True
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.major.size'] = 14
mpl.rcParams['ytick.major.size'] = 14
mpl.rcParams['xtick.minor.size'] = 7
mpl.rcParams['ytick.minor.size'] = 7
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['xtick.minor.width'] = 1.2
mpl.rcParams['ytick.minor.width'] = 1.2
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['xtick.labelsize'] = 20
mpl.rcParams['ytick.labelsize'] = 20
mpl.rcParams['legend.loc'] = 'best'
mpl.rcParams['legend.fontsize'] = 25
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.serif'] = 'Computer Modern'
mpl.rcParams['xtick.major.pad']='8'
mpl.rcParams['ytick.major.pad']='8'
mpl.rcParams['ytick.major.pad']='8'

blue = '#1f77b4'
orange = '#ff7f0e'
green = '#2ca02c'
red = '#ad494a'
violet = '#9467bd'
brown = '#8c564b'

# Test

In [29]:
# This is the function decorator syntax and is equivalent to `hypot = jit(hypot)`.
# The Numba compiler is just a function you can call whenever you want!
@nb.jit
def hypot(x, y):
    # Implementation from https://en.wikipedia.org/wiki/Hypot
    x = abs(x);
    y = abs(y);
    t = min(x, y);
    x = max(x, y);
    t = t / x;
    return x * np.sqrt(1+t*t)

In [21]:
%timeit hypot(3.0, 4.0)
%timeit hypot.py_func(3.0, 4.0)

142 ns ± 0.813 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
1.05 μs ± 7.02 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [34]:
# hypot.inspect_types()

In [60]:
nsamples = 1000000
# TODO: Import Numba's just-in-time compiler function

# TODO: Use the Numba compiler to compile this function
@nb.jit
def monte_carlo_pi(nsamples):
    acc = 0
    for i in range(nsamples):
        x = np.random.random()
        y = np.random.random()
        if (x**2 + y**2) < 1.0:
            acc += 1
    return 4.0 * acc / nsamples

In [13]:
from numba.cuda import cuda
import time

@cuda.jit
def func(x):
    """
    Code for kernel.
    """
    for i in range(x.size):
        x[i] = x[i] * x[i]
    return

def func_nocuda(x):
    """
    Code for kernel.
    """
    for i in range(x.size):
        x[i] = x[i] * x[i]
    return

# Create the data array - usually initialized some other way
data = np.ones(2048 * 2048)    

# Set the number of threads in a block
threadsperblock = 128 

# Calculate the number of thread blocks in the grid
blockspergrid = (data.size + (threadsperblock - 1)) // threadsperblock

print('threadsperblock: ', threadsperblock, 'blockspergrid: ', blockspergrid)


# data transfer
d_data = cuda.to_device(data)

# Now start the kernel
t = time.process_time()
func[blockspergrid, threadsperblock](d_data)
elapsed_time = time.process_time() - t
print('Elapsed time(CUDA): ', elapsed_time)

t = time.process_time()
func_nocuda(data)
elapsed_time = time.process_time() - t
print('Elapsed time(NO CUDA): ', elapsed_time)

threadsperblock:  128 blockspergrid:  32768
Elapsed time(CUDA):  0.046875
Elapsed time(NO CUDA):  0.890625


In [61]:
# This assertion will fail until you successfully complete the exercise one cell above
np.testing.assert_almost_equal(monte_carlo_pi(nsamples), monte_carlo_pi.py_func(nsamples), decimal=2)

%timeit monte_carlo_pi(nsamples)
%timeit monte_carlo_pi.py_func(nsamples)

10.7 ms ± 59 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
863 ms ± 7.81 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
@nb.vectorize([nb.float64(nb.float64, nb.float64)], target='cuda')
def add_ufunc(x, y):
    return x + y

n = 50000
a = np.arange(n, dtype=np.float64)
b = np.arange(n, dtype=np.float64)
c = np.zeros_like(a)

%timeit np.add(a, b)
%timeit add_ufunc(a, b)

19.8 μs ± 206 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


C:\Users\ninoy\AppData\Local\Programs\Python\Python311\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 49 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


898 μs ± 15.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [38]:
@nb.guvectorize([(nb.float64[:], nb.float64[:], nb.float64[:])], '(n),(n)->(n)', target='cuda')
def add_ufunc(x, y, z):
    for i in range(x.size):
        z[i] = x[i] + y[i]

In [39]:
%timeit c = a + b
%timeit add_ufunc(a, b, c)

18.6 μs ± 200 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


C:\Users\ninoy\AppData\Local\Programs\Python\Python311\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


5.49 ms ± 30.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
import numba as nb
from numba.cuda import jit

# @nb.njit()
@jit
def tst_dot():
    a = np.array([[1, 0], [0, 1]], dtype=np.float32)
    b = np.array([[4, 1], [2, 2]], dtype=np.float32)

    return np.dot(a, b)

print(tst_dot[1, 128]())

In [34]:
from numba import cuda, float32
import math

@cuda.jit
def matmul(A, B, C):
    """Perform square matrix multiplication of C = A * B."""
    i, j = cuda.grid(2)
    if i < C.shape[0] and j < C.shape[1]:
        tmp = 0.
        for k in range(A.shape[1]):
            tmp += A[i, k] * B[k, j]
        C[i, j] = tmp

x_h = np.arange(16).reshape([4, 4])
y_h = np.ones([4, 4])
z_h = np.zeros([4, 4])

x_d = cuda.to_device(x_h)
y_d = cuda.to_device(y_h)
z_d = cuda.to_device(z_h)

threadsperblock = (16, 16)
blockspergrid_x = math.ceil(z_h.shape[0] / threadsperblock[0])
blockspergrid_y = math.ceil(z_h.shape[1] / threadsperblock[1])
blockspergrid = (blockspergrid_x, blockspergrid_y)

print(blockspergrid, threadsperblock)

matmul[(1, 1), (16, 16)](x_d, y_d, z_d)
z_h = z_d.copy_to_host()
print(z_h)
print(x_h @ y_h)

(1, 1) (16, 16)
[[ 6.  6.  6.  6.]
 [22. 22. 22. 22.]
 [38. 38. 38. 38.]
 [54. 54. 54. 54.]]
[[ 6.  6.  6.  6.]
 [22. 22. 22. 22.]
 [38. 38. 38. 38.]
 [54. 54. 54. 54.]]


C:\Users\ninoy\AppData\Local\Programs\Python\Python311\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [23]:
# Controls threads per block and shared memory usage.
# The computation will be done on blocks of TPBxTPB elements.
# TPB should not be larger than 32 in this example
TPB = 16

@cuda.jit
def fast_matmul(A, B, C):
    """
    Perform matrix multiplication of C = A * B using CUDA shared memory.

    Reference: https://stackoverflow.com/a/64198479/13697228 by @RobertCrovella
    """
    # Define an array in the shared memory
    # The size and type of the arrays must be known at compile time
    sA = cuda.shared.array(shape=(TPB, TPB), dtype=float32)
    sB = cuda.shared.array(shape=(TPB, TPB), dtype=float32)

    x, y = cuda.grid(2)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    bpg = cuda.gridDim.x    # blocks per grid

    # Each thread computes one element in the result matrix.
    # The dot product is chunked into dot products of TPB-long vectors.
    tmp = float32(0.)
    for i in range(bpg):
        # Preload data into shared memory
        sA[ty, tx] = 0
        sB[ty, tx] = 0
        if y < A.shape[0] and (tx + i * TPB) < A.shape[1]:
            sA[ty, tx] = A[y, tx + i * TPB]
        if x < B.shape[1] and (ty + i * TPB) < B.shape[0]:
            sB[ty, tx] = B[ty + i * TPB, x]

        # Wait until all threads finish preloading
        cuda.syncthreads()

        # Computes partial product on the shared memory
        for j in range(TPB):
            tmp += sA[ty, j] * sB[j, tx]

        # Wait until all threads finish computing
        cuda.syncthreads()
    if y < C.shape[0] and x < C.shape[1]:
        C[y, x] = tmp

In [27]:
x_h = np.arange(16).reshape([4, 4])
y_h = np.ones([4, 4])
z_h = np.zeros([4, 4])

x_d = cuda.to_device(x_h)
y_d = cuda.to_device(y_h)
z_d = cuda.to_device(z_h)

threadsperblock = (TPB, TPB)
blockspergrid_x = math.ceil(z_h.shape[0] / threadsperblock[0])
blockspergrid_y = math.ceil(z_h.shape[1] / threadsperblock[1])
blockspergrid = (blockspergrid_x, blockspergrid_y)

fast_matmul[blockspergrid, threadsperblock](x_d, y_d, z_d)
z_h = z_d.copy_to_host()
print(z_h)
print(x_h @ y_h)

[[ 6.  6.  6.  6.]
 [22. 22. 22. 22.]
 [38. 38. 38. 38.]
 [54. 54. 54. 54.]]
[[ 6.  6.  6.  6.]
 [22. 22. 22. 22.]
 [38. 38. 38. 38.]
 [54. 54. 54. 54.]]
